# Debug an OpenAI Agents workflow with Maida

This tutorial shows how to trace and debug a multi-step OpenAI Agents SDK workflow using Maida. It uses the Agents tracing API to emit deterministic spans, so it runs without API keys or network calls.

## Why agent debugging is hard

AI agents run multi-step workflows with LLM calls, tools, and handoffs between components. When something goes wrong, it is unclear which step failed or why. Maida captures a structured timeline of every run so you can see exactly what happened.

## Setup

If not done already, install Maida with OpenAI Agents support and the OpenAI Agents SDK (run once). If the optional framework dependency is missing, importing the adapter raises an `ImportError` that points back to the `maida-ai[openai]` extra:

In [ ]:
# %pip install --upgrade pip -q
# %pip install "maida-ai[openai]" "openai-agents" -q

In [ ]:
import json
import os
from pathlib import Path
import subprocess
import tempfile

from maida import trace, LoopAbort
from maida.integrations import openai_agents

from agents.tracing import (
    function_span,
    generation_span,
    handoff_span,
    set_trace_processors,
    trace as agents_trace,
)

## Happy-path OpenAI Agents trace

We will model the same **quarterly sales** workflow as the LangGraph tutorial:

- **LLM** decides to search for quarterly sales.
- **Tool** returns deterministic sales data.
- **LLM** decides to calculate the total.
- **Tool** returns the total.
- **LLM** decides to save the result.
- **Tool** confirms saving.
- **LLM** returns a final DONE summary.

Instead of building a graph, we emit OpenAI Agents tracing spans that represent these steps. The Maida OpenAI Agents adapter translates them into `LLM_CALL` and `TOOL_CALL` events.

In [ ]:
@trace(name="OpenAI Agents (happy path)")
def run_openai_agents_happy_path():
    # Tell the OpenAI Agents SDK to send tracing spans to Maida.
    set_trace_processors([openai_agents.PROCESSOR])

    # This block simulates an OpenAI Agents run with deterministic spans.
    with agents_trace("Maida OpenAI Agents happy path"):
        # Agent decides to search.
        with generation_span(
            input=[{"role": "user", "content": "Get quarterly sales and save the total."}],
            output=[{"role": "assistant", "content": "I will search for quarterly sales."}],
            model="gpt-4o-mini",
            model_config={"temperature": 0.0},
            usage={"prompt_tokens": 10, "completion_tokens": 10, "total_tokens": 20},
        ):
            pass

        # search tool returns deterministic quarterly sales.
        with function_span(
            name="search",
            input={"query": "quarterly sales"},
            output={"data": "Q1: 1.2M, Q2: 1.5M, Q3: 2.0M, Q4: 1.8M"},
        ):
            pass

        # Agent decides to calculate the total.
        with generation_span(
            input=[{"role": "assistant", "content": "I will search for quarterly sales."}],
            output=[{"role": "assistant", "content": "I will calculate the total."}],
            model="gpt-4o-mini",
            model_config={"temperature": 0.0},
            usage={"prompt_tokens": 8, "completion_tokens": 8, "total_tokens": 16},
        ):
            pass

        # calculator tool returns the numeric total.
        with function_span(
            name="calculator",
            input={"expression": "1.2+1.5+2.0+1.8"},
            output={"result": 6.5},
        ):
            pass

        # Agent decides to save the result.
        with generation_span(
            input=[{"role": "assistant", "content": "I will calculate the total."}],
            output=[{"role": "assistant", "content": "I will save the result."}],
            model="gpt-4o-mini",
            model_config={"temperature": 0.0},
            usage={"prompt_tokens": 8, "completion_tokens": 8, "total_tokens": 16},
        ):
            pass

        # save_result tool confirms saving.
        with function_span(
            name="save_result",
            input={"data": 6.5},
            output={"result": "Saved: 6.5"},
        ):
            pass

        # Agent returns DONE summary.
        with generation_span(
            input=[{"role": "assistant", "content": "I will save the result."}],
            output=[{"role": "assistant", "content": "DONE. Summary: 6.5M total."}],
            model="gpt-4o-mini",
            model_config={"temperature": 0.0},
            usage={"prompt_tokens": 8, "completion_tokens": 12, "total_tokens": 20},
        ):
            pass


run_openai_agents_happy_path()
print("Run complete. View with: maida view")

### Verify the structural signature

Maida's baseline format extracts the framework-agnostic structure that the pre-merge gate compares. The temporary baseline below verifies the exact offline happy path without checking a run ID into the notebook.

In [ ]:
def capture_latest_baseline() -> dict:
    with tempfile.TemporaryDirectory() as temp_dir:
        output_path = Path(temp_dir) / "baseline.json"
        subprocess.run(
            ["maida", "baseline", "--out", str(output_path)],
            check=True,
            capture_output=True,
            text=True,
        )
        return json.loads(output_path.read_text(encoding="utf-8"))


def structural_signature(baseline: dict) -> dict:
    return {
        "llm_calls": baseline["summary"]["llm_calls"],
        "tool_calls": baseline["summary"]["tool_calls"],
        "tool_call_sequence": baseline["tool_call_sequence"],
        "event_type_sequence": baseline["event_type_sequence"],
        "final_status": baseline["final_status"],
    }

In [ ]:
EXPECTED_SUCCESS_SIGNATURE = {
    "llm_calls": 4,
    "tool_calls": 3,
    "tool_call_sequence": ["search", "calculator", "save_result"],
    "event_type_sequence": [
        "RUN_START",
        "LLM_CALL",
        "TOOL_CALL",
        "LLM_CALL",
        "TOOL_CALL",
        "LLM_CALL",
        "TOOL_CALL",
        "LLM_CALL",
        "RUN_END",
    ],
    "final_status": "ok",
}
success_signature = structural_signature(capture_latest_baseline())
assert success_signature == EXPECTED_SUCCESS_SIGNATURE
print("Verified success signature:", success_signature)

In [ ]:
!maida list

## View the timeline

In a **terminal**, run:

```bash
maida view
```

A browser opens at `http://127.0.0.1:8712`. You will see:

- **Sidebar**: Recent runs (newest first) with name, time, status, duration.
- **Run summary**: Status (OK/ERROR/RUNNING), counts (LLM calls, tools, errors, loop warnings), and filter chips (All, LLM, Tools, Errors, State, Loops).
- **Timeline**: Events in order; expand any event to see payload and meta as JSON.

For this run, the OpenAI Agents adapter turned each **generation span** into an `LLM_CALL` and each **function span** into a `TOOL_CALL`. The meta section contains OpenAI Agents-specific details under `meta.openai_agents.*`. 

### What happened in this run

1. **RUN_START** — Trace began.
2. **LLM_CALL** — Agent decided to search.
3. **TOOL_CALL** (`search`) — Search returned quarterly figures.
4. **LLM_CALL** — Agent decided to calculate.
5. **TOOL_CALL** (`calculator`) — Total 6.5.
6. **LLM_CALL** — Agent decided to save.
7. **TOOL_CALL** (`save_result`) — Saved.
8. **LLM_CALL** — Agent said DONE.
9. **RUN_END** — Trace finished successfully.

This is the same logical sequence as the LangGraph tutorial, but driven by OpenAI Agents tracing spans instead of a graph.

## The looping workflow

Next we simulate an agent **stuck in a loop**: the LLM keeps saying "search again" so the system keeps calling the search tool and re-asking the LLM. We set Maida's loop-detection env vars so a repeated pattern is detected quickly.

In [ ]:
os.environ.setdefault("MAIDA_LOOP_WINDOW", "6")
os.environ.setdefault("MAIDA_LOOP_REPETITIONS", "3")

In [ ]:
@trace(name="OpenAI Agents (looping, no guardrail)")
def run_openai_agents_looping_no_guardrail():
    set_trace_processors([openai_agents.PROCESSOR])

    with agents_trace("Maida OpenAI Agents looping (no guardrail)"):
        # LLM keeps deciding to search again several times.
        for _ in range(5):
            with generation_span(
                input=[{"role": "user", "content": "Find sales"}],
                output=[{"role": "assistant", "content": "I will search again."}],
                model="gpt-4o-mini",
                model_config={"temperature": 0.0},
                usage={"prompt_tokens": 6, "completion_tokens": 6, "total_tokens": 12},
            ):
                pass

            with function_span(
                name="search",
                input={"query": "quarterly sales"},
                output={"data": "Q1: 1.2M, Q2: 1.5M, Q3: 2.0M, Q4: 1.8M"},
            ):
                pass

        # Eventually the agent decides it is done.
        with generation_span(
            input=[{"role": "assistant", "content": "I will search again."}],
            output=[{"role": "assistant", "content": "DONE."}],
            model="gpt-4o-mini",
            model_config={"temperature": 0.0},
            usage={"prompt_tokens": 4, "completion_tokens": 4, "total_tokens": 8},
        ):
            pass


run_openai_agents_looping_no_guardrail()
print("Run finished (no stop_on_loop). Check the timeline for LOOP_WARNING.")

In [ ]:
!maida list

## Understanding LOOP_WARNING

Maida's loop detector looks at the **last N events** (the "window"). If the same **signature pattern** (for example, a `TOOL_CALL` named `search` followed by an `LLM_CALL` from the same provider) repeats **3 times** in a row, it emits a **LOOP_WARNING** event.

In the viewer, use the **Loops** filter to see it; the event payload shows the pattern and which event IDs formed the evidence.

Here the workflow called **search** and the **LLM** three times in a row with no progress. Maida flags that so you can fix the prompt or logic instead of burning more tokens.

## Fix with stop_on_loop

To **prevent** a runaway loop, use the **stop_on_loop** guardrail. When a LOOP_WARNING is emitted and the repetition count meets the minimum, Maida raises **LoopAbort**.

The SDK version locked by this tutorial logs tracing-processor failures at this low-level boundary rather than propagating them to the workflow. The adapter therefore also stores the original guardrail exception on `PROCESSOR.abort_exception`.

As a compatibility fallback for the SDK version locked by this tutorial, **check `PROCESSOR.abort_exception` after each operation** and break early. Then call `PROCESSOR.raise_if_aborted()` so the `@trace` context records the proper `ERROR` and error-status `RUN_END`. Higher-level SDK boundaries or later releases may propagate processor failures differently, so verify the boundary used by your application.

In [ ]:
from maida.integrations.openai_agents import PROCESSOR


@trace(
    name="OpenAI Agents (looping, with guardrail)",
    stop_on_loop=True,
    stop_on_loop_min_repetitions=3,
)
def run_openai_agents_looping_with_guardrail():
    set_trace_processors([openai_agents.PROCESSOR])

    with agents_trace("Maida OpenAI Agents looping (with guardrail)"):
        for i in range(5):
            with generation_span(
                input=[{"role": "user", "content": "Find sales"}],
                output=[{"role": "assistant", "content": "I will search again."}],
                model="gpt-4o-mini",
                model_config={"temperature": 0.0},
                usage={"prompt_tokens": 6, "completion_tokens": 6, "total_tokens": 12},
            ):
                pass

            with function_span(
                name="search",
                input={"query": "quarterly sales"},
                output={"data": "Q1: 1.2M, Q2: 1.5M, Q3: 2.0M, Q4: 1.8M"},
            ):
                pass

            # This locked SDK logs processor failures at the tracing boundary.
            # Poll Maida's captured guardrail exception before the next span.
            if PROCESSOR.abort_exception is not None:
                print(f"  Loop stopped early at iteration {i + 1}")
                break

    PROCESSOR.raise_if_aborted()

try:
    run_openai_agents_looping_with_guardrail()
except LoopAbort as error:
    print(f"Maida stopped the workflow: {error}")
    print("The full trace is saved; open it with: maida view")
else:
    raise AssertionError("Expected the loop guardrail to abort the workflow")

In [ ]:
guarded_signature = structural_signature(capture_latest_baseline())
assert guarded_signature["final_status"] == "error"
assert guarded_signature["event_type_sequence"][-3:] == [
    "LOOP_WARNING", "ERROR", "RUN_END"
]
print("Verified guarded-loop terminal state:", guarded_signature["event_type_sequence"][-3:])

The loop was **stopped early** after iteration 3 (the minimum needed to detect the pattern), rather than completing all five iterations. Its trace ends with `LOOP_WARNING`, `ERROR`, and `RUN_END(status=error)`, preserving the evidence that triggered the guardrail.

For the SDK version locked by this tutorial, this low-level OpenAI Agents tracing path uses the polling fallback because the tracing boundary logs processor failures. In a higher-level integration, check how that boundary propagates failures; where needed, check `PROCESSOR.abort_exception` at a boundary you control and call `PROCESSOR.raise_if_aborted()` before returning success.

## Adapter behavior and failure cases

- **Missing dependency:** importing the OpenAI Agents adapter without the optional SDK raises an `ImportError` with `pip install "maida-ai[openai]"` guidance. Core Maida still imports without OpenAI Agents.
- **No active Maida run:** processor callbacks intentionally record nothing unless SDK spans run inside `@trace`, `traced_run(...)`, or an explicitly enabled implicit run. Register the processor before emitting spans inside the traced entrypoint as shown above.
- **Normalized events:** generation and function spans become Maida's standard `LLM_CALL` and `TOOL_CALL` events; failures use `status="error"`. Framework details stay in metadata instead of adding event types or changing the core schema.
- **Redaction and truncation:** prompts, responses, function arguments, results, errors, and adapter metadata pass through Maida's normal storage protections. Sensitive keys are redacted by default, and oversized fields end with `__TRUNCATED__`. These protections apply to the stored local trace, not to data a real application sends to its model provider.

## Next steps

- Run `maida view` in your terminal to explore the traces from this notebook.
- See the [Maida README](https://github.com/maida-ai/maida#readme) for current guardrails, storage, and integrations.
- Try the LangGraph tutorial notebook to compare traces side by side.